In [1]:
import pandas as pd
import numpy as np
import os
dataset_path = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset"
print(os.listdir(dataset_path))

['data.yaml', 'val', 'test', 'train']


In [2]:
def get_path(path):
    image_path = os.path.join(dataset_path,path,"images")
    label_path = os.path.join(dataset_path,path,"labels")
    return image_path, label_path

train_img,train_lbl = get_path("train")
test_img,test_lbl = get_path("test")
val_img,val_lbl = get_path("val")

In [3]:
train_img_files = [os.path.join(train_img,f) for f in os.listdir(train_img) if f.endswith(('.jpg','.png','.jpeg'))]
test_img_files = [os.path.join(test_img,f) for f in os.listdir(test_img) if f.endswith(('.jpg','.png','.jpeg'))]
val_img_files = [os.path.join(val_img,f) for f in os.listdir(val_img) if f.endswith(('.jpg','.png','.jpeg'))]
train_lbl_files = [os.path.join(train_lbl,f) for f in os.listdir(train_lbl) if f.endswith(('.txt'))]
test_lbl_files = [os.path.join(test_lbl,f) for f in os.listdir(test_lbl) if f.endswith(('.txt'))]
val_lbl_files = [os.path.join(val_lbl,f) for f in os.listdir(val_lbl) if f.endswith(('.txt'))]

In [4]:
print(len(train_img_files),len(train_lbl_files))
print(len(test_img_files),len(test_lbl_files))
print(len(val_img_files),len(val_lbl_files))

8534 8534
1068 1068
1066 1066


In [5]:
import yaml
with open('/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset_path,"train")
data['val'] = os.path.join(dataset_path,"val")
data['test'] =  os.path.join(dataset_path,"test")

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)

In [6]:
!pip -q install ultralytics

In [7]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

2026-01-07 15:51:34.235878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767801094.257130     327 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767801094.263646     327 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767801094.280832     327 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767801094.280853     327 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767801094.280856     327 computation_placer.cc:177] computation placer alr

Num GPUs Available: 2


In [8]:
from ultralytics import YOLO
model = YOLO("yolov8n.yaml")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=30,              
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",      
    lr0=0.003,  
    weight_decay=0.0005,
    cos_lr=True,
    patience=20,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=0.3
)

Ultralytics 8.3.249 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=0.3, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose=12.0,

In [11]:
metrics = results  # results IS DetMetrics

with open("/kaggle/working/metrics.txt", "w") as f:
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"Precision: {metrics.box.p.mean():.4f}\n")
    f.write(f"Recall: {metrics.box.r.mean():.4f}\n")
    f.write(f"F1-score: {metrics.box.f1.mean():.4f}\n")

print("Metrics saved to metrics.txt")


Metrics saved to metrics.txt


In [12]:
model = YOLO("/kaggle/working/runs/detect/train4/weights/best.pt")

model.predict(source="/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test/images",
              conf=0.25,save=True,project="/kaggle/working",name="inference_results")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1068 /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_04_2_600.jpg: 640x640 2 missing_holes, 7.1ms
image 2/1068 /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_07_2_600.jpg: 640x640 2 missing_holes, 7.3ms
image 3/1068 /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_08_1_600.jpg: 640x640 1 missing_hole, 7.2ms
image 4/1068 /kaggle/

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'mouse_bite', 1: 'spur', 2: 'missing_hole', 3: 'short', 4: 'open_circuit', 5: 'spurious_copper'}
 obb: None
 orig_img: array([[[135, 148, 156],
         [132, 145, 153],
         [131, 146, 155],
         ...,
         [127, 148, 140],
         [125, 145, 140],
         [124, 144, 139]],
 
        [[134, 147, 155],
         [131, 144, 152],
         [129, 144, 153],
         ...,
         [136, 156, 151],
         [137, 157, 152],
         [137, 157, 152]],
 
        [[133, 146, 154],
         [131, 144, 152],
         [129, 144, 153],
         ...,
         [131, 150, 147],
         [133, 150, 147],
         [133, 150, 147]],
 
        ...,
 
        [[178, 174, 180],
         [178, 174, 180],
         [179, 175, 181],
         ...,
         [187, 184, 186],
         [187, 184, 186],
         [187, 184, 186]],
 
        [[177, 173, 179

In [13]:
import shutil
import os

source_dir = "/kaggle/working/inference_results"
zip_path = "/kaggle/working/inference_results"

shutil.make_archive(zip_path, 'zip', source_dir)

print("ZIP created at:", zip_path + ".zip")
print("Size:",
      round(os.path.getsize(zip_path + ".zip") / (1024*1024), 2), "MB")


ZIP created at: /kaggle/working/inference_results.zip
Size: 128.78 MB
